In [2]:
import json   
import pandas as pd
import numpy as np
from collections import defaultdict
from indicators import get_indicators_for_token
import os
from openpyxl import load_workbook, Workbook
import glob 
from tqdm import tqdm

In [72]:
# Load JSON file
with open(r"C:\Users\vamsh\Downloads\TA MV2\python_server\ML_Training_datasets\CandleData\Candles\2JBXEbEfzMNZGmP8fRhsUjJQqZUDLhJzwBjvh7vBw7Df_candles.json", "r") as f:
    candle_data = json.load(f)

In [73]:
# Load JSON file
with open(r"C:\Users\vamsh\Downloads\TA MV2\python_server\ML_Training_datasets\CandleData\Stats\2JBXEbEfzMNZGmP8fRhsUjJQqZUDLhJzwBjvh7vBw7Df_stats.json", "r") as f:
    stats_data = json.load(f)

In [74]:
token_name = candle_data.get("name", "Unknown")
candles = candle_data.get("timeframes", {}).get("5S", [])

if not candles:
    print("No candles found in JSON.")
else:
    # Convert to DataFrame
    df = pd.DataFrame(candles)

    # Convert timestamp to datetime column (do NOT set as index)
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    
    # Sort by timestamp just in case
    df = df.sort_values("timestamp").reset_index(drop=True)

    print(f"Token: {token_name}")


Token: Jellybean


In [75]:
df.columns

Index(['timestamp', 'open', 'high', 'low', 'close', 'volume'], dtype='object')

In [76]:
df.head()

,timestamp,open,high,low,close,volume
0,2026-02-27 22:05:45,3.923750e+06,3.926425e+06,3.924924e+06,3.926425e+06,40.206990
1,2026-02-27 22:05:50,3.926425e+06,3.924159e+06,3.919786e+06,3.919786e+06,100.517886
2,2026-02-27 22:06:00,3.919786e+06,3.925031e+06,3.922122e+06,3.925031e+06,78.948570
3,2026-02-27 22:06:05,3.925031e+06,3.981254e+06,3.953001e+06,3.981254e+06,773.203728
4,2026-02-27 22:06:10,3.981254e+06,3.981281e+06,3.977319e+06,3.977319e+06,106.131475


In [77]:
df.shape

(17280, 6)

In [78]:
stats_df = pd.DataFrame(stats_data["stats"])

stats_df["timestamp"] = pd.to_datetime(stats_df["createdAt"])
stats_df = stats_df.sort_values("timestamp")

stats_df = stats_df.drop(columns=["createdAt"])

In [79]:
SUM_COLS = [
    "buyCount",
    "sellCount",
    "buyVolumeSol",
    "sellVolumeSol"
]

MEAN_COLS = ["priceSol"]

In [80]:
WINDOW = 5

stats_5m = stats_df.copy()

stats_5m[SUM_COLS] = (
    stats_df[SUM_COLS]
    .rolling(WINDOW, min_periods=1)
    .sum()
)

stats_5m[MEAN_COLS] = (
    stats_df[MEAN_COLS]
    .rolling(WINDOW, min_periods=1)
    .mean()
)

In [81]:
stats_5m = stats_5m.rename(
    columns={c: f"last_5m_{c}" for c in SUM_COLS + MEAN_COLS}
)

In [82]:
stats_5m.columns

Index(['last_5m_buyCount', 'last_5m_sellCount', 'last_5m_buyVolumeSol',
       'last_5m_sellVolumeSol', 'last_5m_priceSol', 'timestamp'],
      dtype='object')

In [83]:
df["timestamp"] = (
    pd.to_datetime(df["timestamp"], utc=True)
)

stats_5m["timestamp"] = (
    pd.to_datetime(stats_5m["timestamp"], utc=True)
)

In [84]:
df = df.sort_values("timestamp")
stats_5m = stats_5m.sort_values("timestamp")

In [85]:
df_final = pd.merge_asof(
    df,
    stats_5m,
    on="timestamp",
    direction="backward"
)

In [86]:
df_final.shape

(17280, 11)

In [87]:
df_final.head()

,timestamp,open,high,low,close,volume,last_5m_buyCount,last_5m_sellCount,last_5m_buyVolumeSol,last_5m_sellVolumeSol,last_5m_priceSol
0,2026-02-27 22:05:45+00:00,3.923750e+06,3.926425e+06,3.924924e+06,3.926425e+06,40.206990,NaN,NaN,NaN,NaN,NaN
1,2026-02-27 22:05:50+00:00,3.926425e+06,3.924159e+06,3.919786e+06,3.919786e+06,100.517886,NaN,NaN,NaN,NaN,NaN
2,2026-02-27 22:06:00+00:00,3.919786e+06,3.925031e+06,3.922122e+06,3.925031e+06,78.948570,NaN,NaN,NaN,NaN,NaN
3,2026-02-27 22:06:05+00:00,3.925031e+06,3.981254e+06,3.953001e+06,3.981254e+06,773.203728,NaN,NaN,NaN,NaN,NaN
4,2026-02-27 22:06:10+00:00,3.981254e+06,3.981281e+06,3.977319e+06,3.977319e+06,106.131475,NaN,NaN,NaN,NaN,NaN


In [88]:
def flatten_and_align_indicators(ind_dict, N):
    """
    Convert indicators dict to uniform lists of length N.
    Scalars are repeated; lists are padded/truncated to length N.
    Supertrend dicts are flattened.
    Skips 'ema200' indicator.
    """
    aligned = {}
    
    def pad_list(lst, length, fill=np.nan):
        if lst is None: lst = []
        lst = list(lst)
        return [fill]*(length - len(lst)) + lst[-length:] if len(lst) < length else lst[-length:]
    
    for key, val in ind_dict.items():
        if isinstance(val, list):
            if val and isinstance(val[0], dict):
                for subkey in val[0]:
                    aligned[f"{key}_{subkey}"] = pad_list([v[subkey] for v in val], N)
            else:
                aligned[key] = pad_list(val, N)
        else:
            aligned[key] = [val]*N
    return aligned

In [89]:
def list_to_rows(df):
    list_cols = [col for col in [
       'rsi','macd_line','macd_signal','macd_hist','vwap','adx',
       'plus_di','minus_di','stoch_k','stoch_d','boll_upper',
       'boll_middle','boll_lower','atr','obv','supertrend_value',
       'supertrend_trend','ema10','ema20','ema50','ema100','ema200','ema_cross','boll_percent','cci','roc','momentum3'
    ] if col in df.columns]

    scalar_cols = [col for col in df.columns if col not in list_cols]

    # compute max length per row only for existing list columns
    if list_cols:
        max_len_per_row = df[list_cols].applymap(lambda x: len(x) if isinstance(x, list) else 1).max(axis=1)
    else:
        max_len_per_row = pd.Series([1]*len(df))

    def pad_list(lst, length):
        lst = lst if isinstance(lst, list) else [lst]
        return lst + [np.nan]*(length - len(lst))

    # Pad lists and repeat scalars
    for col in df.columns:
        if col in list_cols:
            df[col] = [pad_list(lst, l) for lst, l in zip(df[col], max_len_per_row)]
        else:
            df[col] = [[v]*l for v, l in zip(df[col], max_len_per_row)]

    # Explode all columns
    df_exploded = df.apply(pd.Series.explode).reset_index(drop=True)
    return df_exploded

In [90]:
def append_to_dataset(flattened_indicators, block_df, dataset, distance, extreme_type, pct_change, 
                      current_close, timeframe, high_wick, low_wick, window_size):
    """
    Enrich flattened indicators with additional features and append to dataset.
    Computes time_to_extreme, relative OHLC, volume stats, momentum, and time features.
    """

    # ---------- Time-to-extreme ----------
    flattened_indicators['time_to_extreme'] = distance * timeframe
    flattened_indicators['time_to_extreme_rel'] = distance / window_size  # relative to window length
    flattened_indicators['extreme_type'] = extreme_type
    flattened_indicators['change_percentage'] = pct_change

    # ---------- Wicks ----------
    flattened_indicators['high_wick'] = high_wick
    flattened_indicators['low_wick'] = low_wick

    # ---------- Volume features ----------
    vol = block_df['volume']
    flattened_indicators['volume_sum'] = vol.sum()
    flattened_indicators['volume_avg'] = vol.mean()
    flattened_indicators['volume_rel'] = vol.mean() / vol.iloc[0]
    flattened_indicators['volume_max_rel'] = vol.max() / vol.mean()

    # ---------- High/Low & Range ----------
    flattened_indicators['rel_high'] = block_df['high'].max() / current_close
    flattened_indicators['rel_low'] = block_df['low'].min() / current_close
    flattened_indicators['range_pct'] = (block_df['high'].max() - block_df['low'].min()) / current_close * 100
    flattened_indicators['subblock_len'] = distance

    # ---------- OHLC relative to first candle ----------
    first_candle = block_df.iloc[0]
    flattened_indicators['open_rel'] = first_candle['open'] / current_close
    flattened_indicators['high_rel'] = first_candle['high'] / current_close
    flattened_indicators['low_rel'] = first_candle['low'] / current_close
    flattened_indicators['close_rel'] = first_candle['close'] / current_close

    # ---------- Short-term momentum ----------
    if len(block_df) > 1:
        flattened_indicators['pct_change_1'] = (first_candle['close'] - first_candle['open']) / current_close * 100
        flattened_indicators['pct_change_2'] = (first_candle['close'] - block_df.iloc[1]['close']) / current_close * 100
    else:
        flattened_indicators['pct_change_1'] = 0.0
        flattened_indicators['pct_change_2'] = 0.0

    # ---------- Time features ----------
    ts = pd.to_datetime(first_candle['timestamp'])
    flattened_indicators['hour'] = ts.hour
    flattened_indicators['minute'] = ts.minute
    flattened_indicators['weekday'] = ts.weekday()
    flattened_indicators['timestamp'] = ts

    dataset.append(flattened_indicators)


In [91]:
def balance_memecoin_dataset_full(df, random_state=42):
    """
    Downsamples all classes in 'extreme_type' to match the smallest class size.
    Ensures equal representation of up, down, and stable.

    Parameters
    ----------
    df : pd.DataFrame
        Dataset containing 'extreme_type' column
    random_state : int
        Seed for reproducibility

    Returns
    -------
    pd.DataFrame
        Fully balanced dataset
    """
    counts = df['extreme_type'].value_counts()
    print("Before balancing:\n", counts)

    min_count = counts.min()
    dfs = []

    for label in counts.index:
        sampled = df[df['extreme_type'] == label].sample(n=min_count, random_state=random_state)
        dfs.append(sampled)

    df_balanced = pd.concat(dfs).sample(frac=1, random_state=random_state).reset_index(drop=True)

    print("\nAfter balancing:\n", df_balanced['extreme_type'].value_counts())
    return df_balanced


In [92]:
def add_common_features(base_flattened, subblock, current_close, ts_i): 
    base_flattened["volume_sum"] = subblock["volume"].sum() 
    base_flattened["volume_avg"] = subblock["volume"].mean() 
    base_flattened["range_pct"] = (subblock["high"].max() - subblock["low"].min()) / current_close * 100 
    first_candle = subblock.iloc[0] 
    base_flattened["open_rel"] = first_candle["open"] / current_close 
    base_flattened["high_rel"] = first_candle["high"] / current_close 
    base_flattened["low_rel"] = first_candle["low"] / current_close 
    base_flattened["close_rel"] = first_candle["close"] / current_close 
    base_flattened["hour"] = ts_i.hour 
    base_flattened["minute"] = ts_i.minute 
    base_flattened["weekday"] = ts_i.weekday() 
    base_flattened["timestamp"] = ts_i

In [93]:
def create_forward_label_dataset_clean(df, window_size, step_size,
                                       up_threshold, down_threshold,
                                       stable_threshold, timeframe,
                                       history_bars=300,
                                       target_samples=20,
                                       get_indicators_for_token=get_indicators_for_token):
    """
    For each (subsampled) candle in a sliding window, create up to 3 rows:
      - one for "up"  if price ever rises >= up_threshold%
      - one for "down" if price ever drops <= -down_threshold%
      - one for "stable" if price never hits up/down and
        stays within +/- stable_threshold% over the horizon.

    Indicators are computed once per candle's history and reused for each label row.
    """
    dataset = []

    df_closes = df["close"].values
    max_horizon = int((2 * 3600) / (timeframe * 60))  # 2 hours worth of candles
    df_len = len(df)
    start_idx = 0

    total_windows = ((df_len - window_size) // step_size) + 1

    with tqdm(total=total_windows, desc="Windows") as pbar:

        while start_idx + window_size <= df_len:
            block_df = df.iloc[start_idx:start_idx + window_size].copy()
            closes = block_df["close"].values
            N = len(closes)
          
            half = target_samples // 2
            all_idx = np.arange(0, N - 1)
            
            # evenly spaced anchors
            anchor_idx = (
                np.linspace(0, N - 2, half, dtype=int)
                if half > 0 else np.array([], dtype=int)
            )
            
            # remaining pool after anchors
            remaining = np.setdiff1d(all_idx, anchor_idx)
            
            num_random = min(target_samples - len(anchor_idx), len(remaining))
            
            random_idx = (
                np.random.choice(remaining, size=num_random, replace=False)
                if num_random > 0 else np.array([], dtype=int)
            )
            
            sample_indices = np.unique(np.concatenate([anchor_idx, random_idx]))
    
            np.random.shuffle(sample_indices)
    
            for i in sample_indices:
                abs_i = start_idx + i
                current_close = df_closes[abs_i]
                ts_i = pd.to_datetime(df.iloc[abs_i]["timestamp"])
    
                # Track first occurrence of each label
                up_found = False
                down_found = False
                up_distance = down_distance = None
                up_pct_change = down_pct_change = None
    
                max_abs_change = 0.0
                last_change_pct = 0.0
    
                # Compute horizon bound as an index
                horizon_limit = min(abs_i + max_horizon, df_len - 1)
    
                for abs_j in range(abs_i + 1, horizon_limit + 1):
                    future_close = df_closes[abs_j]
                    change_pct = (future_close - current_close) / current_close * 100.0
                    dist = abs_j - abs_i
    
                    max_abs_change = max(max_abs_change, abs(change_pct))
                    last_change_pct = change_pct
    
                    if not up_found and change_pct >= up_threshold:
                        up_found = True
                        up_distance = dist
                        up_pct_change = change_pct
    
                    if not down_found and change_pct <= -down_threshold:
                        down_found = True
                        down_distance = dist
                        down_pct_change = change_pct
    
                    # Once we have both, no need to look further
                    if up_found and down_found:
                        break
    
                # Stable only if NO up/down happened over the whole horizon
                stable_found = False
                stable_distance = None
                stable_pct_change = None
    
                if not up_found and not down_found and stable_threshold is not None:
                    if max_abs_change <= stable_threshold:
                        stable_found = True
                        stable_distance = horizon_limit - abs_i
                        stable_pct_change = last_change_pct
    
                # If none of the labels triggered, skip this candle
                if not (up_found or down_found or stable_found):
                    continue
    
                # Build history subblock (no leakage)
                start_sub = max(0, abs_i - history_bars + 1)
                subblock = df.iloc[start_sub:abs_i + 1].copy()
                if len(subblock) < history_bars:
                    continue
    
                # Compute indicators once
                indicators = get_indicators_for_token(subblock.to_dict("records"))
                base_flattened = flatten_and_align_indicators(indicators, len(subblock))
    
                # Add common features once to base
                add_common_features(base_flattened, subblock, current_close, ts_i)
    
                # Helper to append a row for each label
                def add_row(label_name, dist, pct_change):
                    row = base_flattened.copy()
                    row["extreme_type"] = label_name
                    row["time_to_extreme"] = dist * timeframe
                    row["pct_change"] = pct_change
                    row["distance"] = dist
                    row["current_close"] = current_close
                    dataset.append(row)
    
                # Create up to 3 rows for this candle
                if up_found and up_distance is not None:
                    add_row("up", up_distance, up_pct_change)
    
                if down_found and down_distance is not None:
                    add_row("down", down_distance, down_pct_change)
    
                if stable_found and stable_distance is not None:
                    add_row("stable", stable_distance, stable_pct_change)
    
            start_idx += step_size
            pbar.update(1)

    df_dataset = pd.DataFrame(dataset)
    df_dataset.fillna(np.nan, inplace=True)
    return df_dataset


In [94]:
dataset = create_forward_label_dataset_clean(
    df=df_final,
    window_size=400,
    step_size=50,
    up_threshold=30.0,
    down_threshold=25.0,
    stable_threshold=5.0,
    timeframe=5,
    history_bars=300,
    target_samples=100,
    get_indicators_for_token=get_indicators_for_token
)


Windows: 100%|███████████████████████████████████████████████████████████████████████| 338/338 [28:33<00:00,  5.07s/it]


In [95]:
exploded_dataset = list_to_rows(dataset)

MemoryError: Unable to allocate 2.19 GiB for an array with shape (37, 7961400) and data type object

In [64]:
exploded_dataset.shape

(0, 0)

In [ ]:
counts = exploded_dataset['extreme_type'].value_counts()
print(counts)

In [ ]:
exploded_dataset.columns

In [ ]:
exploded_dataset.head()

In [ ]:
exploded_dataset_clean = exploded_dataset.dropna(how="any").reset_index(drop=True)

In [ ]:
exploded_dataset_clean.shape

In [ ]:
counts = exploded_dataset_clean['extreme_type'].value_counts()
print(counts)

In [ ]:
exploded_dataset_clean.head()

In [ ]:
df_new = balance_memecoin_dataset_full(exploded_dataset_clean)

In [ ]:
df_new.shape

In [ ]:
df_new['time_to_extreme'].describe()

In [ ]:
df_new.head()

In [ ]:
with pd.ExcelWriter(r"C:\Users\vamsh\Downloads\TA MV2\python_server\ML_Training_datasets\memecoin_training_dataset.xlsx") as writer:
    df_new.to_excel(writer, sheet_name="5s_candles")

In [13]:
folder_path = r"C:\Users\vamsh\Downloads\TA MV2\python_server\ML_Training_datasets\CandleData"
base_csv_path = r"C:\Users\vamsh\Downloads\TA MV2\python_server\ML_Training_datasets\memecoin_training_dataset_5S.csv"
sheet_name = "5s_candles"  # unused now, but fine if you keep it

json_files = [f for f in os.listdir(folder_path) if f.endswith(".json")]
batch_size = 5
file_number = 1

for batch_idx, i in enumerate(range(0, len(json_files), batch_size)):
    batch_files = json_files[i:i+batch_size]

    # Name for this batch CSV
    csv_file_path = base_csv_path.replace(".csv", f"_{batch_idx+1}.csv")

    for file_name in batch_files:
        file_path = os.path.join(folder_path, file_name)
        with open(file_path, "r") as f:
            data = json.load(f)

        token_name = data.get("name", "Unknown")
        candles = data.get("timeframes", {}).get("5S", [])

        if not candles:
            print(f"No candles found in {file_name}. Skipping.")
            continue

        df = pd.DataFrame(candles)
        if "timestamp" in df.columns:
            df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")

        df = df.sort_values("timestamp").reset_index(drop=True)

        print(f"{file_number}, Processing token: {token_name}, file: {file_name}")

        token_id = os.path.splitext(file_name)[0]

        dataset = create_forward_label_dataset_clean(
            df=df,
            window_size=400,
            step_size=50,
            up_threshold=30.0,
            down_threshold=25.0,
            stable_threshold=5.0,
            timeframe=5,
            history_bars=300,
            target_samples=100,
            get_indicators_for_token=get_indicators_for_token
        )

        print(f"  -> dataset rows: {len(dataset)}")

        exploded_dataset = list_to_rows(dataset)
        print(f"  -> exploded rows: {len(exploded_dataset)}")

        exploded_dataset["token_id"] = token_id

        # Skip if only 'stable' exists
        unique_types = exploded_dataset["extreme_type"].unique()
        
        if len(unique_types) == 1 and unique_types[0] == "stable":
            print("  -> only 'stable' labels found, skipping CSV append\n")
            file_number += 1
            continue


        exploded_dataset_clean = exploded_dataset.dropna(how="any").reset_index(drop=True)
        print(f"  -> clean rows: {len(exploded_dataset_clean)}")

        df_new = balance_memecoin_dataset_full(exploded_dataset_clean)
        print(f"  -> Balanced rows: {len(df_new)}")

        print("  -> writing to CSV...")
        df_new.to_csv(
            csv_file_path,
            mode="a",
            header=not os.path.exists(csv_file_path),
            index=False
        )
        print("  -> done writing\n")

        file_number += 1


1, Processing token: Rizzoween, file: 2dCYTTCQxPsMZeBR2b3x3wmKYytfxsnsEtyy7nwGQzEU.json


Windows: 100%|█████████████████████████████████████████████████████████████████████████| 65/65 [02:08<00:00,  1.97s/it]


  -> dataset rows: 1876
  -> exploded rows: 562800
  -> clean rows: 538412
Before balancing:
 stable    507129
up         15785
down       15498
Name: extreme_type, dtype: int64

After balancing:
 stable    15498
up        15498
down      15498
Name: extreme_type, dtype: int64
  -> Balanced rows: 46494
  -> writing to CSV...
  -> done writing

2, Processing token: PHI, file: 2RKZJaD7sSues6YxoujsafcBYjxW6t4h9AVgiRWyDjdw.json


Windows: 100%|█████████████████████████████████████████████████████████████████████████| 65/65 [06:13<00:00,  5.74s/it]


  -> dataset rows: 5967
  -> exploded rows: 1790100
  -> only 'stable' labels found, skipping CSV append

3, Processing token: AIONIX, file: 2XKwu8fJTrV22UjH8jThejxtctHDCspiubdEmbx2A4dv.json


...

  -> dataset rows: 1633
  -> exploded rows: 489900
  -> clean rows: 468671
Before balancing:
 up        197743
down      151249
stable    119679
Name: extreme_type, dtype: int64

After balancing:
 down      119679
up        119679
stable    119679
Name: extreme_type, dtype: int64
  -> Balanced rows: 359037
  -> writing to CSV...
  -> done writing

27, Processing token: Tika, file: qASXryCs5Vz5XYf2BhgNqchAicFjp7no7BovQ3ggkep.json


Windows: 100%|███████████████████████████████████████████████████████████████████████████| 9/9 [00:18<00:00,  2.05s/it]


  -> dataset rows: 305
  -> exploded rows: 91500
  -> clean rows: 87535
Before balancing:
 down    52521
up      35014
Name: extreme_type, dtype: int64

After balancing:
 up      35014
down    35014
Name: extreme_type, dtype: int64
  -> Balanced rows: 70028
  -> writing to CSV...
  -> done writing

28, Processing token: FARTLESS, file: UKSPrYDU4veB4cAfV9HkzNg9gNP1jsoch8oySMG9tcJ.json


Windows: 100%|█████████████████████████████████████████████████████████████████████████| 65/65 [05:42<00:00,  5.27s/it]


  -> dataset rows: 5555
  -> exploded rows: 1666500
  -> only 'stable' labels found, skipping CSV append



In [14]:
# required_columns = ['rsi', 'ema10', 'ema20', 'ema50', 'ema100', 'ema200', 'ema_cross',
#        'macd_line', 'macd_signal', 'macd_hist', 'vwap', 'adx', 'plus_di',
#        'minus_di', 'stoch_k', 'stoch_d', 'boll_upper', 'boll_middle',
#        'boll_lower', 'boll_percent', 'atr', 'obv', 'supertrend_value',
#        'supertrend_trend', 'cci', 'roc', 'momentum3', 'extreme_type',
#        'time_to_extreme', 'change_percentage', 'volume_sum', 'volume_avg',
#        'volume_rel', 'volume_max_rel', 'fixed_h_pct_change',
#        'fixed_h_direction', 'rel_high', 'rel_low', 'range_pct', 'subblock_len',
#        'open_rel', 'high_rel', 'low_rel', 'close_rel', 'pct_change_1',
#        'pct_change_2', 'hour', 'minute', 'weekday', 'timestamp']

# Read and concatenate all files

import os
import pandas as pd

folder_path = r"C:\Users\vamsh\Downloads\TA MV2\python_server\ML_Training_datasets"
output_file = os.path.join(folder_path, "memecoin_training_dataset_5S.csv")

csv_files = [
    os.path.join(folder_path, f)
    for f in os.listdir(folder_path)
    if f.endswith(".csv") and f != os.path.basename(output_file)
]

total_rows = 0
first = True

with open(output_file, "w", newline="", encoding="utf-8") as out:
    for file in csv_files:
        print(f"Appending {file}")
        for chunk in pd.read_csv(file, chunksize=200_000):
            chunk.to_csv(out, header=first, index=False)
            total_rows += len(chunk)
            first = False

print(f"\n✅ Combined {len(csv_files)} files")
print(f"✅ Total rows: {total_rows}")
print(f"✅ Saved to: {output_file}")



Appending C:\Users\vamsh\Downloads\TA MV2\python_server\ML_Training_datasets\memecoin_training_dataset_5S_1.csv
Appending C:\Users\vamsh\Downloads\TA MV2\python_server\ML_Training_datasets\memecoin_training_dataset_5S_2.csv
Appending C:\Users\vamsh\Downloads\TA MV2\python_server\ML_Training_datasets\memecoin_training_dataset_5S_3.csv
Appending C:\Users\vamsh\Downloads\TA MV2\python_server\ML_Training_datasets\memecoin_training_dataset_5S_4.csv
Appending C:\Users\vamsh\Downloads\TA MV2\python_server\ML_Training_datasets\memecoin_training_dataset_5S_5.csv
Appending C:\Users\vamsh\Downloads\TA MV2\python_server\ML_Training_datasets\memecoin_training_dataset_5S_6.csv

✅ Combined 6 files
✅ Total rows: 1190476
✅ Saved to: C:\Users\vamsh\Downloads\TA MV2\python_server\ML_Training_datasets\memecoin_training_dataset_5S.csv
